# Kalman Filter — Two-Generator Power System State Estimation

| | |
|---|---|
| **Author** | Marin Dagron |
| **Date** | January 2026 |
| **Context** | Industrial Project — CentraleSupélec |

---

## Objective

This notebook implements a **discrete-time Kalman Filter** to reconstruct the full dynamic state of a two-generator power system from noisy measurements, in the presence of various load disturbances.

The approach follows four main steps:

1. **Nonlinear simulation** — establish a ground-truth reference using the full nonlinear model.
2. **Linearization** — derive the state-space matrices $(A, B, C, D)$ around the operating equilibrium using symbolic Jacobians.
3. **Observability analysis** — verify that the system can be fully reconstructed from the chosen output set.
4. **Kalman filtering** — run the predict/correct loop on noisy measurements and evaluate estimation quality.

---

## Notebook Structure

```
0. Imports & Configuration
1. System Parameters
2. Disturbance Scenarios
3. Nonlinear Model & Simulation
4. Linearization (Symbolic)
5. Observability Analysis
6. Linear Model Simulation (Euler)
7. Kalman Filter
8. Results & Visualization
```

---
## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from scipy.integrate import solve_ivp
from scipy.linalg import expm
from tqdm import tqdm

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.grid': True,
    'grid.alpha': 0.4,
    'lines.linewidth': 1.5,
    'font.size': 11,
})

print('All imports successful.')

---
## 1. System Parameters

All physical parameters for the two-machine network are centralised in a single dictionary.
Changing any value here propagates automatically to every subsequent step.

| Symbol | Description | Unit |
|--------|-------------|------|
| $\omega_0$ | Nominal angular frequency | rad/s |
| $K_L$ | Line synchronising coefficient | MW/rad |
| $K_s$ | Secondary control gain | — |
| $J_i$ | Inertia constant of generator $i$ | — |
| $D_i$ | Damping coefficient of generator $i$ | — |
| $\alpha_i, \beta_i$ | Primary governor gains | — |
| $P_{r,i}$ | Participation factor (secondary control) | MW |

In [ ]:
PARAMS = {
    # Network
    'f0'     : 50.0,
    'omega0' : 2 * np.pi * 50.0,   # rad/s
    'KL'     : 3064.0,             # MW/rad  (two parallel lines)
    'Ks'     : 0.05,               # secondary control gain

    # Generator 1
    'g1': {
        'P0'  : 600.0,   # nominal active power setpoint   [MW]
        'Pmin': 0.0,     # minimum power output            [MW]
        'Pmax': 1000.0,  # maximum power output            [MW]
        'Pr'  : 100.0,   # secondary participation factor  [MW]
        'J'   : 0.4,     # inertia constant
        'D'   : 0.04,    # damping coefficient
        'alpha': 100.0,  # governor proportional gain
        'beta' : 2000.0, # governor frequency gain
        'PL'  : 400.0,   # local load                      [MW]
    },

    # Generator 2
    'g2': {
        'P0'  : 400.0,
        'Pmin': 0.0,
        'Pmax': 600.0,
        'Pr'  : 50.0,
        'J'   : 0.1,
        'D'   : 0.02,
        'alpha': 100.0,
        'beta' : 2000.0,
        'PL'  : 600.0,
    },

    # Simulation
    't_event': 15.0,
    # Available scenarios: 'EL1', 'EL2', 'SL1', 'SL2', 'PLT', 'Random'
    'scenario': 'Random',
}

print(f"Nominal frequency  : {PARAMS['f0']} Hz")
print(f"Line stiffness KL  : {PARAMS['KL']} MW/rad")
print(f"Active scenario    : {PARAMS['scenario']}")

---
## 2. Disturbance Scenarios

Six disturbance profiles are available:

| ID | Description |
|----|-------------|
| `EL1` | Linear load ramp: +100 MW over 60 s on G1 |
| `EL2` | Linear load ramp: +200 MW over 60 s on G1 |
| `SL1` | Load step at $t = 15$ s → $P_{L1} = 500$ MW |
| `SL2` | Load step at $t = 15$ s → $P_{L1} = 600$ MW |
| `PLT` | Line trip at $t = 15$ s: $K_L$ halved |
| `Random` | Independent 1 % Gaussian noise on both loads |

In [ ]:
def generate_colored_noise(n_steps: int, dt: float, tau: float =10.0, sigma: float=5.0, seed=None):
    """
    Génère un bruit corrélé temporellement via un processus AR(1).
    
    tau   : constante de temps de corrélation [s]
          (plus tau est grand, plus le bruit varie lentement)
    sigma : écart-type du bruit stationnaire
    """
    rng   = np.random.default_rng(seed)
    alpha = np.exp(-dt / tau)          # coefficient d'autocorrélation
    noise = np.zeros(n_steps)
    for k in tqdm(range(1, n_steps)):
        noise[k] = alpha * noise[k-1] + np.sqrt(1 - alpha**2) * sigma * rng.normal()
    return noise



In [ ]:
D_T =1e-5
N_LIN = int(20/1e-5)

pl1_noise = generate_colored_noise(N_LIN, D_T, tau=50.0, sigma=25.0)
pl2_noise = generate_colored_noise(N_LIN, D_T, tau=50.0, sigma=25.0)

def get_disturbances(t: float, scenario: str, params: dict):
    """
    Return instantaneous values of (p1l, p2l, kl) for a given scenario.

    Parameters
    ----------
    t        : current simulation time [s]
    scenario : disturbance profile identifier
    params   : global parameter dictionary

    Returns
    -------
    p1l : load on G1 bus [MW]
    p2l : load on G2 bus [MW]
    kl  : line synchronising coefficient [MW/rad]
    """
    p1l = params['g1']['PL']   # 400 MW (baseline)
    p2l = params['g2']['PL']   # 600 MW (baseline)
    kl  = params['KL']         # 3064 MW/rad (baseline)

    if scenario == 'EL1':
        # +100 MW linear ramp over 60 s
        p1l = 400.0 + (100.0 / 60.0) * min(t, 60.0)

    elif scenario == 'EL2':
        # +200 MW linear ramp over 60 s
        p1l = 400.0 + (200.0 / 60.0) * min(t, 60.0)

    elif scenario == 'SL1':
        if t >= params['t_event']:
            p1l = 500.0

    elif scenario == 'SL2':
        if t >= params['t_event']:
            p1l = 600.0

    elif scenario == 'PLT':
        # Line trip: synchronising coefficient halved
        if t >= params['t_event']:
            kl = 1532.0

    elif scenario == 'BBAG':
        rng = np.random.default_rng()
        p1l = p1l * (1.0 + rng.normal(0, 0.01))
        p2l = p2l * (1.0 + rng.normal(0, 0.01))
    
    elif scenario == 'Random':
        # Add colored noise to loads
        i = int(t / 1e-5)  # Convert time to index for noise arrays
        if i < len(pl1_noise):
            p1l += pl1_noise[i]
            p2l += pl2_noise[i]

    else:
        raise ValueError(f"Unknown scenario: '{scenario}'. "
                         "Choose from EL1, EL2, SL1, SL2, PLT, BBAG, Random.")

    return p1l, p2l, kl

---
## 3. Nonlinear Model & Simulation

### State vector

$$\mathbf{x} = [\theta_1,\; \omega_1,\; T_{m1},\; \theta_2,\; \omega_2,\; T_{m2},\; N]^\top$$

### Governing equations

**Line power:**
$$F_{12} = K_L \sin(\theta_1 - \theta_2)$$

**Swing equation (generator $i$):**
$$J_i\, \dot{\omega}_i = T_{mi} - \frac{P_{Gi}}{\omega_i} - D_i(\omega_i - \omega_0)$$

**Primary governor:**
$$\dot{T}_{mi} = -\alpha_i(T_{mi}\omega_i - P_{ci}) - \beta_i(\omega_i - \omega_0), \quad P_{ci} = \text{clip}(P_{0i} + N P_{ri})$$

**Secondary control:**
$$\dot{N} = -K_s\left(\frac{J_1\omega_1 + J_2\omega_2}{J_1+J_2} - \omega_0\right)$$

In [ ]:
def power_system_ode(t: float, x: list, params: dict) -> list:
    """
    Full nonlinear ODE of the two-generator power system.

    State:  x = [theta1, omega1, Tm1, theta2, omega2, Tm2, N]
    Returns dx/dt.
    """
    theta1, omega1, Tm1, theta2, omega2, Tm2, N = x
    p1l, p2l, kl = get_disturbances(t, params['scenario'], params)
    w0 = params['omega0']

    # ── Algebraic equations ───────────────────────────────────────────────────
    F12  = kl * np.sin(theta1 - theta2)   # power flow on tie-line [MW]
    P1G  = p1l + F12                       # demand seen by G1
    P2G  = p2l - F12                       # demand seen by G2
    P1m  = Tm1 * omega1                    # actual mechanical power G1
    P2m  = Tm2 * omega2                    # actual mechanical power G2

    # Setpoints (secondary correction included, with output limits)
    P1c = np.clip(params['g1']['P0'] + N * params['g1']['Pr'],
                  params['g1']['Pmin'], params['g1']['Pmax'])
    P2c = np.clip(params['g2']['P0'] + N * params['g2']['Pr'],
                  params['g2']['Pmin'], params['g2']['Pmax'])

    # ── Differential equations ────────────────────────────────────────────────
    d_theta1 = omega1 - w0
    d_omega1 = (Tm1 - P1G / omega1 - params['g1']['D'] * (omega1 - w0)) / params['g1']['J']

    if params['g1']['Pmin'] <= P1m <= params['g1']['Pmax']:
        d_Tm1 = -params['g1']['alpha'] * (P1m - P1c) - params['g1']['beta'] * (omega1 - w0)
    else:
        d_Tm1 = 0.0

    d_theta2 = omega2 - w0
    d_omega2 = (Tm2 - P2G / omega2 - params['g2']['D'] * (omega2 - w0)) / params['g2']['J']

    if params['g2']['Pmin'] <= P2m <= params['g2']['Pmax']:
        d_Tm2 = -params['g2']['alpha'] * (P2m - P2c) - params['g2']['beta'] * (omega2 - w0)
    else:
        d_Tm2 = 0.0

    omega_avg = (params['g1']['J'] * omega1 + params['g2']['J'] * omega2) / \
                (params['g1']['J'] + params['g2']['J'])
    dN = -params['Ks'] * (omega_avg - w0) if -1.0 <= N <= 1.0 else 0.0

    return [d_theta1, d_omega1, d_Tm1, d_theta2, d_omega2, d_Tm2, dN]

In [ ]:
# ── Equilibrium initial conditions ────────────────────────────────────────────
w0 = PARAMS['omega0']
KL = PARAMS['KL']

theta2_eq = -np.arcsin((PARAMS['g1']['P0'] - PARAMS['g1']['PL']) / KL)
x0_nl = [                             # Equilibrium initial conditions
    0.0,                              # theta1
    w0,                               # omega1
    PARAMS['g1']['P0'] / w0,          # Tm1
    theta2_eq,                        # theta2
    w0,                               # omega2
    PARAMS['g2']['P0'] / w0,          # Tm2
    0.0,                              # N
]


# ── Solve ODE ─────────────────────────────────────────────────────────────────
T_NL   = 20.0
D_T = 1e-4  # time step [s]
N_EVAL = int(T_NL / D_T)+1
t_eval_nl = np.linspace(0, T_NL, N_EVAL)

sol_nl = solve_ivp(
    power_system_ode,
    (0, T_NL),
    x0_nl,
    args=(PARAMS,),
    t_eval=t_eval_nl,
    method='RK45',
    rtol=1e-6, atol=1e-9,
)


df_X_simul = pd.DataFrame(sol_nl.y.T, columns=['theta1', 'omega1', 'Tm1', 'theta2', 'omega2', 'Tm2', 'N'])  
df_X_simul['time'] = t_eval_nl
df_X_simul = df_X_simul.set_index('time')


'theta1', 'theta2', 'f12', 'pg1', 'pg2', 'pl1', 'pl2'

df_Y_simul = pd.DataFrame({
    'theta1' : df_X_simul['theta1'],
    'theta2' : df_X_simul['theta2'],
    'f12': KL * np.sin(df_X_simul['theta1'] - df_X_simul['theta2']),
    'pl1': [get_disturbances(t, PARAMS['scenario'], PARAMS)[0] for t in df_X_simul.index],  # Initial load (constant for display consistency)
    'pl2': [get_disturbances(t, PARAMS['scenario'], PARAMS)[1] for t in df_X_simul.index]   # Initial load (constant for display consistency)
}, index=df_X_simul.index) 

df_Y_simul['pg1'] = df_Y_simul['pl1'] + df_Y_simul['f12']  # Power generated by G1
df_Y_simul['pg2'] = df_Y_simul['pl2'] - df_Y_simul['f12']  # Power generated by G2

# df_X_simul.index = pd.to_timedelta(df_X_simul.index, unit='s') 
# df_Y_simul.index = pd.to_timedelta(df_Y_simul.index, unit='s')

# ── Derived signals ───────────────────────────────────────────────────────────
th1_nl = sol_nl.y[0]
th2_nl = sol_nl.y[3]
F12_nl = KL * (th1_nl - th2_nl)   # linearised for display consistency

print(f"Solver status  : {sol_nl.message}")
print(f"Number of steps: {sol_nl.t.size}")

In [ ]:

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_X_simul.index, y=df_X_simul["omega1"] / (2 * np.pi),  name='Omega1',  line=dict(color='green', width=1.5)))
# fig.add_trace(go.Scatter(x=df_X_simul.index, y=df_X_simul["omega2"] / (2 * np.pi),  name='Omega2',  line=dict(color='silver', width=1.0)))


fig.update_layout(
    title=dict(text='Frequency Response simulation', font=dict(size=13)),
    xaxis_title='Time [s]',
    yaxis_title='Frequency [rad/s]',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.12),
    height=350)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_Y_simul.index, y=df_Y_simul["pl1"],  name='pl1',  line=dict(color='green', width=1.5)))
# fig.add_trace(go.Scatter(x=df_Y_simul.index, y=df_Y_simul["pl2"],  name='pl2',  line=dict(color='silver', width=1.0)))


fig.update_layout(
    title=dict(text='Load', font=dict(size=13)),
    xaxis_title='Time [s]',
    yaxis_title='Load [MW]',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.12),
    height=350)
fig.show()

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# fig.suptitle('Nonlinear Simulation — System Response', fontsize=14, fontweight='bold')

# ax = axes[0, 0]
# ax.plot(sol_nl.t, sol_nl.y[1] / (2 * np.pi), label='G1')
# ax.plot(sol_nl.t, sol_nl.y[4] / (2 * np.pi), label='G2')
# ax.axhline(PARAMS['f0'], color='grey', linestyle='--', label='50 Hz')
# ax.set_ylabel('Frequency [Hz]');  ax.set_title('Frequency'); ax.legend()

# ax = axes[0, 1]
# ax.plot(sol_nl.t, sol_nl.y[0], label='$\\theta_1$')
# ax.plot(sol_nl.t, sol_nl.y[3], label='$\\theta_2$')
# ax.set_ylabel('Angle [rad]');  ax.set_title('Rotor Angles'); ax.legend()

# ax = axes[1, 0]
# ax.plot(sol_nl.t, sol_nl.y[2] * sol_nl.y[1], label='G1')
# ax.plot(sol_nl.t, sol_nl.y[5] * sol_nl.y[4], label='G2')
# ax.set_ylabel('Power [MW]'); ax.set_title('Mechanical Power'); ax.legend()

# ax = axes[1, 1]
# ax.plot(sol_nl.t, F12_nl, color='darkorange')
# ax.set_ylabel('Power [MW]'); ax.set_title('Tie-line Power $F_{12}$')

# for ax in axes.flat:
#     ax.set_xlabel('Time [s]')
# plt.tight_layout()
# plt.show()

---
## 4. Linearization (Symbolic)

The nonlinear system is linearised around the equilibrium point $\mathbf{x}_{eq}$
using **SymPy automatic differentiation** (Jacobians).

The load disturbances $p_{l1}$ and $p_{l2}$ are treated as **augmented states**
(with $\dot{p}_{li} = 0$), enabling the Kalman filter to jointly estimate
system states and unknown loads.

### Extended state vector (9 states)

$$\mathbf{x} = [\theta_1,\; \omega_1,\; T_{m1},\; \theta_2,\; \omega_2,\; T_{m2},\; N,\; p_{l1},\; p_{l2}]^\top$$

### Linear state-space form

$$\dot{\mathbf{x}} = A\mathbf{x} + B\mathbf{u} + D\mathbf{w}, \qquad \mathbf{y} = C\mathbf{x} + E\mathbf{w}$$

where $\mathbf{u} = [P_{0,1},\, P_{0,2}]^\top$ and $\mathbf{w} = [p_{l1},\, p_{l2}]^\top$.

In [ ]:
def build_linear_model(params: dict):
    """
    Linearise the power system around the operating equilibrium.

    Returns
    -------
    X_sym : SymPy state vector (9 x 1)
    Y_sym : SymPy output vector (7 x 1)
    A, B, C, D, E : numeric numpy arrays
    """
    # ── Symbolic variables ────────────────────────────────────────────────────
    t1, w1, tm1, t2, w2, tm2, n = sp.symbols('theta1 omega1 Tm1 theta2 omega2 Tm2 N')
    p10, p20, pl1, pl2         = sp.symbols('P10 P20 P1L P2L')
    kl, w0, j1, j2             = sp.symbols('KL omega0 J1 J2')
    d1, d2, a1, a2, b1, b2     = sp.symbols('D1 D2 alpha1 alpha2 beta1 beta2')
    pr1, pr2, ks               = sp.symbols('Pr1 Pr2 Ks')

    X_sym = sp.Matrix([t1, w1, tm1, t2, w2, tm2, n, pl1, pl2])

    # ── Linearised dynamics (sin ≈ θ₁ − θ₂ near equilibrium) ────────────────
    f12  = kl * (t1 - t2)
    pg1  = pl1 + f12
    pg2  = pl2 - f12

    f = sp.Matrix([
        w1 - w0,
        (tm1 - pg1 / w0 - d1 * (w1 - w0)) / j1,
        -a1 * (tm1 * w0 - (p10 + n * pr1)) - b1 * (w1 - w0),
        w2 - w0,
        (tm2 - pg2 / w0 - d2 * (w2 - w0)) / j2,
        -a2 * (tm2 * w0 - (p20 + n * pr2)) - b2 * (w2 - w0),
        -ks * ((j1 * w1 + j2 * w2) / (j1 + j2) - w0),
        sp.Integer(0),
        sp.Integer(0),
    ])

    Y_sym = sp.Matrix([t1, t2, f12, pg1, pg2, pl1, pl2])

    # ── Jacobians ─────────────────────────────────────────────────────────────
    A_sym = f.jacobian(X_sym)
    B_sym = f.jacobian(sp.Matrix([p10, p20]))
    D_sym = f.jacobian(sp.Matrix([pl1, pl2]))
    C_sym = Y_sym.jacobian(X_sym)
    E_sym = Y_sym.jacobian(sp.Matrix([pl1, pl2]))

    # ── Numerical substitution ────────────────────────────────────────────────
    subs = {
        w0: params['omega0'], kl: params['KL'], ks: params['Ks'],
        p10: params['g1']['P0'],  pl1: params['g1']['PL'],
        j1:  params['g1']['J'],   d1:  params['g1']['D'],
        a1:  params['g1']['alpha'], b1: params['g1']['beta'],
        pr1: params['g1']['Pr'],
        p20: params['g2']['P0'],  pl2: params['g2']['PL'],
        j2:  params['g2']['J'],   d2:  params['g2']['D'],
        a2:  params['g2']['alpha'], b2: params['g2']['beta'],
        pr2: params['g2']['Pr'],
    }

    to_num = lambda M: np.array(M.subs(subs)).astype(np.float64)

    return X_sym, Y_sym, to_num(A_sym), to_num(B_sym), to_num(C_sym), \
           to_num(D_sym), to_num(E_sym)


X_sym, Y_sym, A_num, B_num, C_num, D_num, E_num = build_linear_model(PARAMS)
E_num = np.zeros_like(E_num)  # Ignore direct disturbance-to-output feedthrough for now

print(f"State dimension  : {A_num.shape[0]}")
print(f"Output dimension : {C_num.shape[0]}")
print(f"Disturbance inputs (pl1, pl2): {D_num}")
print("\nMatrix A:\n", np.round(A_num, 4))
print("\nMatrix B:\n", np.round(B_num, 4))
print("\nMatrix C:\n", np.round(C_num, 4))
print("\nMatrix D:\n", np.round(D_num, 4))
print("\nMatrix E:\n", np.round(E_num, 4))

---
## 5. Observability Analysis

The **observability matrix** is:
$$\mathcal{O} = \begin{bmatrix} C \\ CA \\ CA^2 \\ \vdots \\ CA^{n-1} \end{bmatrix} \in \mathbb{R}^{(n_y \cdot n) \times n}$$

The system is **fully observable** if and only if $\text{rank}(\mathcal{O}) = n$.

In [ ]:
def check_observability(A: np.ndarray, C: np.ndarray, tol: float = 1e-8) -> dict:
    """
    Compute the observability matrix and its rank.

    Parameters
    ----------
    A   : system matrix (n x n)
    C   : output matrix (p x n)
    tol : rank tolerance

    Returns
    -------
    dict with keys: 'O', 'rank', 'n', 'observable'
    """
    n = A.shape[0]
    O = np.vstack([C @ np.linalg.matrix_power(A, k) for k in range(n)])
    rank = np.linalg.matrix_rank(O, tol=tol)
    return {'O': O, 'rank': rank, 'n': n, 'observable': rank == n}


obs = check_observability(A_num, C_num)

print(f"Observability matrix shape : {obs['O'].shape}")
print(f"Rank                       : {obs['rank']} / {obs['n']}")
print()
if obs['observable']:
    print("The system is FULLY OBSERVABLE — Kalman filter can reconstruct all states.")
else:
    unobs = obs['n'] - obs['rank']
    print(f"The system is NOT fully observable ({unobs} unobservable mode(s)).")
    print("    Artificial process noise will be injected on augmented load states")
    print("    to regularise the filter (see Section 7).")

---
## 6. Linear Model Simulation (Euler Forward)

The linearised model is integrated with the **Euler forward** method to produce the
ground-truth reference trajectories used by the Kalman filter.

$$\Delta\mathbf{x}_{k+1} = \Delta\mathbf{x}_k + \bigl(A\,\Delta\mathbf{x}_k + B\,\Delta\mathbf{u} + D\,\Delta\mathbf{w}\bigr)\,\Delta t$$

> **Note:** A small time step ($\Delta t = 10^{-5}$ s) is required for numerical stability
> of the explicit Euler scheme given the stiff eigenvalues of $A$.

In [ ]:
# ── Equilibrium point ─────────────────────────────────────────────────────────
theta2_eq = -np.arcsin((PARAMS['g1']['P0'] - PARAMS['g1']['PL']) / PARAMS['KL'])

# A_num = expm(A_num * 1e-5)  # Scale A for numerical stability in Euler integration

x_eq = np.array([
    0.0,
    PARAMS['omega0'],
    PARAMS['g1']['P0'] / PARAMS['omega0'],
    theta2_eq,
    PARAMS['omega0'],
    PARAMS['g2']['P0'] / PARAMS['omega0'],
    0.0,
    PARAMS['g1']['PL'],
    PARAMS['g2']['PL'],
])

u_eq = np.array([PARAMS['g1']['P0'], PARAMS['g2']['P0']])
w_eq = np.array([PARAMS['g1']['PL'], PARAMS['g2']['PL']])

# ── Time grid ─────────────────────────────────────────────────────────────────
DT_LIN  = 1e-5     # Euler time step [s]
T_LIN   = 10.0      # total duration  [s]
t_steps = np.arange(0.0, T_LIN, DT_LIN)
N_LIN   = len(t_steps)
n_x     = len(x_eq)
n_y     = C_num.shape[0]

# ── Storage ───────────────────────────────────────────────────────────────────
x_hist  = np.empty((N_LIN, n_x))
y_hist  = np.empty((N_LIN, n_y))

delta_x = np.zeros(n_x)   # starts at equilibrium → delta = 0
x_hist[0] = x_eq
y_hist[0] = C_num @ x_eq + E_num @ w_eq

delta_u = np.zeros(2)

pl1_noise = generate_colored_noise(N_LIN, DT_LIN, tau=50.0, sigma=25.0)
pl2_noise = generate_colored_noise(N_LIN, DT_LIN, tau=50.0, sigma=25.0)

print(f"Running Euler integration: {N_LIN:,} steps at dt = {DT_LIN} s ...")

for i in tqdm(range(1, N_LIN)):
    #p1l, p2l, _ = get_disturbances(t_steps[i], PARAMS['scenario'], PARAMS)
    p1l = w_eq[0] + pl1_noise[i]
    p2l = w_eq[1] + pl2_noise[i]
    delta_w = np.array([p1l, p2l]) - w_eq

    ddx = A_num @ delta_x + B_num @ delta_u # + D_num @ delta_w
    delta_x = delta_x + ddx * DT_LIN

    x_hist[i] = delta_x + x_eq
    y_hist[i] = C_num @ x_hist[i] #+ E_num @ np.array([p1l, p2l])

# ── Convert to DataFrames ─────────────────────────────────────────────────────
STATE_COLS  = ['theta1', 'omega1', 'Tm1', 'theta2', 'omega2', 'Tm2', 'N', 'pl1', 'pl2']
OUTPUT_COLS = ['theta1', 'theta2', 'f12', 'pg1', 'pg2', 'pl1', 'pl2']

df_X = pd.DataFrame(x_hist, columns=STATE_COLS)
# df_X['omega1'] = df_X['omega1'] / (2 * np.pi)   # rad/s → Hz
# df_X['omega2'] = df_X['omega2'] / (2 * np.pi)
df_X.index = pd.to_timedelta(t_steps, unit='s')

df_Y = pd.DataFrame(y_hist, columns=OUTPUT_COLS)
df_Y.index = pd.to_timedelta(t_steps, unit='s')

# ── Save reference data ───────────────────────────────────────────────────────
# df_X.to_csv(f'Ref_States_{PARAMS["scenario"]}.csv')
# df_Y.to_csv(f'Ref_Outputs_{PARAMS["scenario"]}.csv')

print("\nLinear simulation complete.")
print(f"  df_X shape: {df_X.shape}  |  df_Y shape: {df_Y.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle(f'Linear Simulation — Scenario: {PARAMS["scenario"]}',
             fontsize=13, fontweight='bold')

axes[0, 0].plot(t_steps, df_Y['theta1'].values, label='$\\theta_1$')
axes[0, 0].plot(t_steps, df_Y['theta2'].values, label='$\\theta_2$')
axes[0, 0].set_title('Rotor Angles'); axes[0, 0].set_ylabel('rad'); axes[0, 0].legend()

axes[0, 1].plot(t_steps, df_Y['pl1'].values, label='Load 1')
axes[0, 1].plot(t_steps, df_Y['pl2'].values, label='Load 2')
axes[0, 1].set_title('Load Disturbances'); axes[0, 1].set_ylabel('MW'); axes[0, 1].legend()

axes[1, 0].plot(t_steps, df_Y['f12'].values, color='darkorange')
axes[1, 0].set_title('Tie-line Power $F_{12}$'); axes[1, 0].set_ylabel('MW')

axes[1, 1].plot(t_steps, df_Y['pg1'].values, color='purple')
axes[1, 1].plot(t_steps, df_Y['pg2'].values, color='teal')
axes[1, 1].set_title('Power seen by G1 $P_{1G}$ and G2 $P_{2G}$')
axes[1, 1].set_ylabel('MW')

for ax in axes.flat:
    ax.set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

---
## 7. Kalman Filter

### Discretisation

The continuous-time matrices are discretised via the **matrix exponential** at
the filter time step $\Delta t_K$:

$$A_d = e^{A\,\Delta t_K}, \quad B_d = B\,\Delta t_K, \quad D_d = D\,\Delta t_K$$

### Filter equations

**Predict:**
$$\hat{\mathbf{x}}^-_{k} = A_d\,\hat{\mathbf{x}}_{k-1} + B_d\,\Delta\mathbf{u} + D_d\,\Delta\mathbf{w} + \mathbf{v}_k$$
$$P^-_k = A_d P_{k-1} A_d^\top + Q$$

**Correct:**
$$\mathbf{e}_k = \Delta\mathbf{y}_k - C\hat{\mathbf{x}}^-_k$$
$$S_k = C P^-_k C^\top + R$$
$$K_k = P^-_k C^\top S_k^{-1}$$
$$\hat{\mathbf{x}}_k = \hat{\mathbf{x}}^-_k + K_k\,\mathbf{e}_k$$
$$P_k = (I - K_k C)\,P^-_k$$

### Tuning note on $Q$

The augmented load states $p_{l1}$, $p_{l2}$ have $\dot{p}_{li} = 0$ in the model,
making them unobservable through the state dynamics alone.
A **deliberate process noise** $\sigma_w = 5$ is injected on those two channels of $Q$,
signalling to the filter that these states are uncertain and must be corrected via measurements.

In [ ]:
# ── Discretisation ────────────────────────────────────────────────────────────
DT_K = 1e-4   # Kalman filter time step [s]  (coarser than Euler step)
Ad   = expm(A_num * DT_K)          # matrix exponential
Bd   = B_num * DT_K
Dd   = D_num * DT_K
C_k  = C_num
E_k  = np.zeros_like(E_num)        # direct feedthrough neglected

# ── Resample reference outputs to Kalman time grid ───────────────────────────
df_Y_simul.index = pd.to_timedelta(df_Y_simul.index, unit='s')
df_Y_simul = df_Y_simul.resample(f'{DT_K}s').interpolate(method='linear')

# ── Measurement noise (1 % of mean signal amplitude per channel) ──────────────
sigma_meas = np.abs(0.01 * df_Y_simul.abs().max().values)
sigma_meas = np.where(sigma_meas < 1e-12, 1e-6, sigma_meas)  # avoid zero
noise_matrix = np.random.normal(0.0, sigma_meas, df_Y_simul.shape)
df_Y_noisy = df_Y_simul + noise_matrix
df_Y_noisy = df_Y_noisy[['theta1', 'theta2', 'f12', 'pg1', 'pg2', 'pl1', 'pl2']]

plt.figure(figsize=(10, 4))
plt.plot(df_Y_simul.index, df_Y_simul['pl1'], label='Clean pl1', color='green')
plt.plot(df_Y_noisy.index, df_Y_noisy['pl1'], label='Noisy pl1', color='lightgreen', alpha=0.6)
plt.title('Measurement Noise on Load pl1')
plt.xlabel('Time [s]')
plt.ylabel('Load [MW]')
plt.legend()
plt.show()

print("Measurement noise std (σ) per channel:")
for col, s in zip(OUTPUT_COLS, sigma_meas):
    print(f"  {col:8s}: {s:.4e}")

# ── Noise covariance matrices ─────────────────────────────────────────────────
Q_base = np.diag([
    1e-6,   # θ₁  [rad²]
    1e-6,   # ω₁  [rad²/s²]   ← was 1.0, far too large
    1e-4,   # Tm1
    1e-6,   # θ₂
    1e-6,   # ω₂              ← same
    1e-4,   # Tm2
    1e-6,   # N
    1e-4,   # pl1  [MW²]
    1e-4,   # pl2  [MW²]
])

R_k = np.diag(sigma_meas**2)   # measurement noise covariance

# ── Initialisation ────────────────────────────────────────────────────────────
x_hat    = np.zeros((n_x, 1))   # start at zero 
x_eq[3]  = theta2_eq                # set correct equilibrium angle for theta2
x_eq_vec = np.array(x_eq).reshape(n_x, 1)
x_hat    = np.random.normal(0, 0.00, (n_x, 1)) * x_eq_vec # small random offset from equilibrium

P_cov    = np.eye(n_x) * 0.1
y_eq_vec = (C_k @ x_eq ).reshape(-1, 1)

t_K        = df_Y_noisy.index.total_seconds().values
N_K        = len(t_K)
xhat_hist  = np.empty((N_K, n_x))
yhat_hist  = np.empty((N_K, n_y))

du = np.zeros((2, 1))
dw = np.zeros((2, 1))

# ── Filter loop ───────────────────────────────────────────────────────────────
print(f"\nRunning Kalman filter: {N_K:,} steps ...")
P_eigvalues = pd.DataFrame(columns=[f'λ_{i}' for i in STATE_COLS ], index=t_K)
Trace_P = []
for i in tqdm(range(N_K)):

    delta_y = df_Y_noisy.iloc[i].values.reshape(-1, 1) - y_eq_vec

    # -- Predict ---------------------------------------------------------------
    x_hat_minus = Ad @ x_hat + Bd @ du # + Dd @ dw
    P_minus     = Ad @ P_cov @ Ad.T + Q_base # compute prior covariance (process noise added here)

    # -- Correct ---------------------------------------------------------------
    innovation  = delta_y - C_k @ x_hat_minus
    S           = C_k @ P_minus @ C_k.T + R_k # innovation covariance
    K           = P_minus @ C_k.T @ np.linalg.inv(S)

    x_hat = x_hat_minus + K @ innovation
    P_cov = (np.eye(n_x) - K @ C_k) @ P_minus # 
    
    # IKC = np.eye(n_x) - K @ C_k
    # P_cov = IKC @ P_minus @ IKC.T + K @ R_k @ K.T # always symmetric PD 
    
    P_eigvalues.iloc[i] = np.linalg.eigvals(P_cov)
    
    Trace_P.append(np.trace(P_cov))

    xhat_hist[i] = x_hat.flatten()
    yhat_hist[i] = (C_k @ x_hat  + y_eq_vec).flatten()

# ── Build output DataFrames ───────────────────────────────────────────────────
df_X_hat = pd.DataFrame(xhat_hist + x_eq, columns=STATE_COLS)
df_X_hat['omega1'] = df_X_hat['omega1'] / (2 * np.pi)
df_X_hat['omega2'] = df_X_hat['omega2'] / (2 * np.pi)
df_X_hat['time']   = t_K

df_Y_hat = pd.DataFrame(yhat_hist, columns=OUTPUT_COLS)
df_Y_hat['time'] = t_K

print("\nKalman filter complete.")

eigvals_P_cov = np.linalg.eigvals(P_cov)
trace_P_cov = np.trace(P_cov)
print(f"\nEigenvalues of P_cov:\n{eigvals_P_cov}")
print(f"Final error covariance trace: {trace_P_cov:.4e}")

plt.plot(t_K[5:], Trace_P[5:])
plt.title('Trace of Estimation Error Covariance $P_{cov}$ over Time')
plt.xlabel('Time [s]')
plt.ylabel('Trace($P_{cov}$)')
plt.grid()
plt.show()

print(f"\nKalman gain K:\n{K}")


In [ ]:
fig, axes = plt.subplots(1, figsize=(14, 8))
fig.suptitle('Covariance Analysis', fontsize=14, fontweight='bold')
axes.plot(t_K, P_eigvalues)
axes.set_title('Eigenvalues of $P_{cov}$')
axes.set_xlabel('Time [s]')
axes.legend(P_eigvalues.columns)
axes.set_ylabel('Eigenvalue')

# --- Ajoute cette ligne pour borner le temps entre 0 et 3 secondes ---
axes.set_xlim(0, 3)

axes.set_yscale('log')
axes.grid()
plt.tight_layout()
plt.show()

---
## 8. Results & Visualisation

Each plot shows three curves:
- 🟩 **Model** — linear simulation reference (ground truth)
- ◼ **Noisy** — simulated sensor measurements
- 🟥 **Kalman** — filter estimate

The Signal-to-Noise Ratio (SNR) is also reported for each estimated output channel.

In [ ]:

def compute_snr(signal: np.ndarray, estimate: np.ndarray) -> float:
    """
    Compute SNR [dB] = 10 log10( var(signal) / var(signal - estimate) ).
    A higher value indicates a closer estimate to the true signal.
    """
    noise_power  = np.var(signal - estimate)
    signal_power = np.var(signal)
    if noise_power < 1e-30:
        return np.inf
    return 10 * np.log10(signal_power / noise_power)



def plot_comparison(time, y_model, y_noisy, y_kalman, title, ylabel, snr=None):
    """Plot model vs noisy vs Kalman estimate with optional SNR annotation."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time, y=y_noisy,  name='Noisy',  line=dict(color='silver', width=1.0)))
    fig.add_trace(go.Scatter(x=time, y=y_kalman, name='Kalman', line=dict(color='red',   width=1.5)))
    fig.add_trace(go.Scatter(x=time, y=y_model,  name='Simulation',  line=dict(color='green', width=1.5)))

    snr_text = f'  |  SNR = {snr:.1f} dB' if snr is not None else ''
    fig.update_layout(
        title=dict(text=title + snr_text, font=dict(size=13)),
        xaxis_title='Time [s]',
        yaxis_title=ylabel,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', y=1.12),
        height=350,
    )
    fig.show()

In [ ]:
# ── Align all data on the Kalman time grid ────────────────────────────────────
df_X_lin_k = df_X.resample(f'{DT_K}s').interpolate(method='linear')
df_Y_lin_k = df_Y.resample(f'{DT_K}s').interpolate(method='linear')

time_k = t_K   # seconds

In [ ]:
# ── Tie-line power F12 ────────────────────────────────────────────────────────
snr_f12 = compute_snr(df_Y_simul['f12'].values, df_Y_hat['f12'].values)

plot_comparison(
    time=time_k,
    y_model  = df_Y_simul['f12'].values,
    y_noisy  = df_Y_noisy['f12'].values,
    y_kalman = df_Y_hat['f12'].values,
    title='Tie-line Power F12',
    ylabel='Flux [MW]',
    snr=snr_f12,
)

In [ ]:
# ── Generated power G1 ────────────────────────────────────────────────────────
snr_pg1 = compute_snr(df_Y_simul['pg1'].values, df_Y_hat['pg1'].values)

plot_comparison(
    time=time_k,
    y_model  = df_Y_simul['pg1'].values,
    y_noisy  = df_Y_noisy['pg1'].values,
    y_kalman = df_Y_hat['pg1'].values,
    title='Generated Power PG1',
    ylabel='Power [MW]',
    snr=snr_pg1,
)

In [ ]:
# ── Load disturbance estimation (pl1) ─────────────────────────────────────────
snr_pl1 = compute_snr(df_Y_simul['pl1'].values, df_Y_hat['pl1'].values)

plot_comparison(
    time=time_k,
    y_model  = df_Y_simul['pl1'].values,
    y_noisy  = df_Y_noisy['pl1'].values,
    y_kalman = df_Y_hat['pl1'].values,
    title='Load Disturbance Estimation PL1',
    ylabel='Load [MW]',
    snr=snr_pl1,
)

In [ ]:
# ── Angular frequency G1 (state) ──────────────────────────────────────────────
snr_w1 = compute_snr(df_X_simul['omega1'].values/(2 * np.pi), df_X_hat['omega1'].values)

plot_comparison(
    time=time_k,
    y_model  = df_X_simul['omega1'].values/(2 * np.pi),
    y_noisy  = df_X_simul['omega1'].values/(2 * np.pi) + np.random.normal(0, sigma_meas[0], N_K),
    y_kalman = df_X_hat['omega1'].values,
    title='Angular Frequency omega_1 (G1)',
    ylabel='Frequency [Hz]',
    snr=snr_w1,
)

In [ ]:
# ── Secondary control signal N (state) ───────────────────────────────────────
snr_theta = compute_snr(df_X_simul['theta1'].values, df_Y_hat['theta1'].values)

plot_comparison(
    time=time_k,
    y_model  = df_X_simul['theta1'].values,
    y_noisy  = df_X_simul['theta1'].values + np.random.normal(0, sigma_meas[0], N_K),
    y_kalman = df_Y_hat['theta1'].values,
    title='Angular Displacement theta1 (G1)',
    ylabel='p.u.',
    snr=snr_theta,
)

In [ ]:
# ── Secondary control signal N (state) ───────────────────────────────────────
snr_theta = compute_snr(df_X_simul['theta2'].values, df_Y_hat['theta2'].values)

plot_comparison(
    time=time_k,
    y_model  = df_X_simul['theta2'].values,
    y_noisy  = df_X_simul['theta2'].values + np.random.normal(0, sigma_meas[0], N_K),
    y_kalman = df_Y_hat['theta2'].values,
    title='Angular Displacement theta2 (G2)',
    ylabel='p.u.',
    snr=snr_theta,
)

In [ ]:
# ── Secondary control signal N (state) ───────────────────────────────────────
snr_F12 = compute_snr(PARAMS['KL']*(-df_X_simul['theta2'].values + df_X_simul['theta1'].values), PARAMS['KL']*(-df_Y_hat['theta2'].values + df_Y_hat['theta1'].values))


plot_comparison(
    time=time_k,
    y_model  = PARAMS['KL']*(-df_X_simul['theta2'].values + df_X_simul['theta1'].values),
    y_noisy  = PARAMS['KL']*(-df_X_simul['theta2'].values + df_X_simul['theta1'].values + np.random.normal(0, sigma_meas[0], N_K)),
    y_kalman = PARAMS['KL']*(-df_Y_hat['theta2'].values + df_Y_hat['theta1'].values),
    title='Flux Linkage (proportional to angle difference)',
    ylabel='MW',
    snr=snr_F12,
)

In [ ]:
# ── Secondary control signal N (state) ───────────────────────────────────────
snr_N = compute_snr(df_X_simul['N'].values, df_X_hat['N'].values)

plot_comparison(
    time=time_k,
    y_model  = df_X_simul['N'].values,
    y_noisy  = df_X_simul['N'].values + np.random.normal(0, 1e-4, N_K),
    y_kalman = df_X_hat['N'].values,
    title='Secondary Control Signal $N$',
    ylabel='p.u.',
    snr=snr_N,
)

In [ ]:
# ── Performance Summary ──────────────────────────────────────────────────────
print("="*55)
print(f"  PERFORMANCE SUMMARY (%) — Scenario: {PARAMS['scenario']}")
print("="*55)

snr_results = {}

# 1. Boucle sur toutes les variables d'état (X)
x_cols = [c for c in df_X_hat.columns if c in df_X_simul.columns and c != 'time']
for col in x_cols:
    err_std = np.std(df_X_hat[col].values - df_X_simul[col].values)
    mean_val = np.abs(np.mean(df_X_simul[col].values))
    mean_val = mean_val if mean_val > 1e-9 else 1.0  # Protection division par zéro
    snr_results[f"X_{col}"] = (err_std / mean_val) * 100

# 2. Boucle sur toutes les variables de sortie (Y)
y_cols = [c for c in df_Y_hat.columns if c in df_Y_simul.columns and c != 'time']
for col in y_cols:
    err_std = np.std(df_Y_hat[col].values - df_Y_simul[col].values)
    mean_val = np.abs(np.mean(df_Y_simul[col].values))
    mean_val = mean_val if mean_val > 1e-9 else 1.0  # Protection division par zéro
    snr_results[f"Y_{col}"] = (err_std / mean_val) * 100

# 3. Affichage du tableau
print(f"  {'Signal':<10s} | {'Erreur (%)':<10s} | Graphique")
print("-" * 55)
for signal, err_val in snr_results.items():
    # Échelle d'affichage de la barre (1 bloc = 0.5% d'erreur, max 30 blocs)
    bar_length = min(int(err_val * 2), 30) if np.isfinite(err_val) else 0
    bar = '█' * bar_length if bar_length > 0 else '|'
    
    print(f"  {signal:<10s} | {err_val:8.4f} %  {bar}")
print("="*55)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1. Time Parameters ────────────────────────────────────────────────────────
T_MAX = 20.0       # Total simulation time [s]
T_BREAK = 2.0      # Instant of the line trip [s]
dt = 1e-3          # Time step for display
t = np.arange(0, T_MAX, dt)

# ── 2. Tie-Line Parameters (KL) ───────────────────────────────────────────────
KL_init = 3064.0         # Initial synchronising coefficient [MW/rad]
KL_post = KL_init / 2.0  # 50% drop after the fault (one conductor trips)

# ── 3. KL Profile Creation ────────────────────────────────────────────────────
# np.where acts as a vectorized "if/else" over the entire time array
KL_trace = np.where(t < T_BREAK, KL_init, KL_post)

# ── 4. Plotting the Figure ────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(t, KL_trace, color='firebrick', linewidth=2, label='$K_L$ Dynamics')

# Vertical line to clearly mark the fault instant
plt.axvline(x=T_BREAK, color='grey', linestyle='--', alpha=0.7, label=f'Line Trip ($t={T_BREAK}$s)')

plt.title('Evolution of the Tie-Line Coupling Coefficient $K_L$ Following a Line Trip', fontsize=12)
plt.xlabel('Time [s]')
plt.ylabel('Coefficient $K_L$ [MW/rad]')

# Limits to properly frame the graph
plt.xlim(0, T_MAX)
plt.ylim(0, KL_init * 1.2) 

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()